In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [2]:
def search(query: str) -> list:
    """Search the documentation database for relevant results.

    Args:
        query: The search query to look up in the index.

    Returns:
        The top five matching results from the documentation database.
    """
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [3]:
def add_entry(
    filename: str,
    title: str,
    description: str,
    content: str
) -> str:
    """Add a new documentation entry to the index.

    Args:
        filename: The source filename associated with the entry.
        title: The title of the documentation entry.
        description: A short description summarizing the entry.
        content: The full content of the documentation entry.

    Returns:
        "OK" after the entry has been added to the index.
    """
    entry = {
        "start": 0,
        "content": content,
        "title": title,
        "description": description,
        "filename": filename,
    }
    index.append(entry)
    return "OK"

In [4]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [5]:
from toyaikit.tools import Tools
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner

In [6]:
from toyaikit.tools import Tools
agent_tools = Tools()

agent_tools.add_tool(search)
agent_tools.add_tool(add_entry)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results.\n\nArgs:\n    query: The search query to look up in the index.\n\nReturns:\n    The top five matching results from the documentation database.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.\n\nArgs:\n    filename: The source filename associated with the entry.\n    title: The title of the documentation entry.\n    description: A short description summarizing the entry.\n    content: The full content of the documentation entry.\n\nReturns:\n    "OK" after the entry has been added to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'},
    'title'

In [7]:
from openai  import OpenAI

In [8]:
llm_client = OpenAIClient(model="gpt-4o-mini", client=OpenAI())

In [9]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [10]:
agent= OpenAIResponsesRunner(tools=agent_tools,developer_prompt=instructions,
                              chat_interface=chat_interface,llm_client=llm_client)

In [11]:
result = agent.loop('how can I create evidently dahsbord?',callback=runner_callback)


-> Response received


-> Response received


-> Response received


In [12]:
class SearchIndexTools:
     def __init__(self, index):
        self.index = index
     def search(self,query: str) -> list:
      """Search the documentation database for relevant results.

      Args:
         query: The search query to look up in the index.

      Returns:
         The top five matching results from the documentation database.
      """
      results = index.search(
         query=query,
         num_results=5
      )
      return results
     def add_entry(
         filename: str,
         title: str,
         description: str,
         content: str
      ) -> str:
         """Add a new documentation entry to the index.

         Args:
            filename: The source filename associated with the entry.
            title: The title of the documentation entry.
            description: A short description summarizing the entry.
            content: The full content of the documentation entry.

         Returns:
            "OK" after the entry has been added to the index.
         """
         entry = {
            "start": 0,
            "content": content,
            "title": title,
            "description": description,
            "filename": filename,
         }
         index.append(entry)
         return "OK"
         
            

In [13]:
search_tools_object = SearchIndexTools(index)

results = search_tools_object.search('dashboard')
results

[{'start': 0,
  'content': 'You can add Panels in the user interface or using Python API. This pages describes the Python API. Check how to [add panels in the UI](dashboard_add_panels_ui).\n\n## Dashboard Management\n\n<Check>\n  Dashboards as code are available in Evidently OSS, Cloud, Enterprise.\n</Check>\n\n<Tip>\n  You must first connect to [Evidently Cloud](/docs/setup/cloud) and [create a Project](/docs/platform/projects_manage).\n</Tip>\n\n**Adding Tabs**. To add a new Tab:\n\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\nYou can also create a new Tab while adding a Panel as shown below. If the destination Tab doesn\'t exist, it will be created. If it does, the Panel will be added below existing ones in that Tab.\n\n**Deleting Tabs**. To delete a Tab:\n\n```python\nproject.dashboard.delete_tab("Another Tab")\n```\n\n**Deleting Panels**. To delete a specific Panel:\n\n```python\nproject.dashboard.delete_panel("Dashboard title", "My new tab")\n```\n\n(First list the

In [14]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
     chat_interface=chat_interface,
    llm_client=llm_client
)

In [15]:
result = agent.loop('how can I create evidently dahsbord?')

In [18]:
print(result.last_message)

Based on the gathered information from the searches, here’s how to create a dashboard effectively:

### Creating a Dashboard Overview

1. **Dashboard Management**:
   - To set up a dashboard, first connect to your project.
   - You can create multiple tabs to organize your panels, and each tab can display different insights.
   - **Adding Tabs**: You can add tabs in edit mode, either from the UI or with the command:
     ```python
     project.dashboard.add_tab("New Tab")
     ```

2. **Adding Panels**:
   - Panels are crucial for visualizing data. You can add various panel types such as counters, text panels, plots, and distributions.
   - **Adding a Panel**: In edit mode, click "Add Panel" and follow the prompts to configure.
   - From the API, use:
     ```python
     project.dashboard.add_panel(DashboardPanelPlot(...), tab="My Tab")
     ```

### Types of Panels

- **Counter Panels**: Used for metric counts or displaying simple metrics. Example:
  ```python
  project.dashboard.add_

In [42]:
result.cost.input_cost


Decimal('0.00261225')

In [43]:
result.cost.output_cost

Decimal('0.0004374')

In [44]:
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit.chat.interface import IPythonChatInterface

In [45]:
chat_interface = IPythonChatInterface()

In [46]:
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [47]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [28]:
result2= agent.loop('show me the code',callback=runner_callback,previous_messages=result.all_messages)

-> Response received


In [34]:
agent= OpenAIResponsesRunner(tools=agent_tools,developer_prompt=instructions,chat_interface=chat_interface,llm_client=llm_client)


In [ ]:
result = agent.run()
stop